# Schizophrenia Pathway Classifier — Enrichment Analysis

## Objective

In this notebook I use enrichment analysis to see if immune/inflammatory pathway genes show expression differences in SCZ samples compared to control samples. To do this, I run differential expression (DE) testing then apply False Discovery Rate (FDR) correction, afterward I rank the gene list, then lastly use Gene Set Enrichment Analysis (GSEA) against the immune/inflammatory gene set. This answers the first part of my primary hypothesis: whether immune/inflammatory pathway genes, implicated in schizophrenia by independent studies, show enrichment in expression differences between SCZ vs. control subjects in this dataset.

## Input
- `../data/processed/merged_df.csv`

## 2.1 Setup & Load Data

Import most of the same libraries from EDA, with the addition of `scipy` for statistical testing, `statsmodels` for FDR, and `gseapy` for GSEA. Additionally, I set random state and absolute path to `merged_df.csv` for reproducibility. I read in the `merged_df.csv` since it contains both the expression values and metadata needed for the downstream analyses. 

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse 

from scipy import stats
import statsmodels.stats.multitest as smt
import gseapy as gp

sns.set_theme()
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
PROCESSED_DATA_PATH = '../data/processed/merged_df.csv'

In [14]:
merged_df = pd.read_csv(PROCESSED_DATA_PATH)
# check shape and value counts to confirm df was imported correctly
print(merged_df.shape)
merged_df['diagnosis'].value_counts() 

(59, 30065)


diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64

Correct shape (59, 30065) and class balance (30/29 SCZ samples vs. control)

## 2.2 Probe-to-Gene Mapping

I translate probe IDs to gene symbols, since the `merged_df`'s columns are currently Affymetrix probe IDs, so that I can compare immune/inflammatory pathway list against my ranked list created downstream for GSEA. To do this I load GPL570 using `GEOparse.get_GEO()`, and then use that to produce a lookup dictionary that maps probes to genes. 

To map probes to genes I first load `GPL570` using `GEOparse.get_GEO()`, then I confirm shape and look at the columns to identify which contains gene symbols. Once I've identified the correct column containing gene symbols, I check for any duplicates or missing values, since multiple probes (duplicates) can map to the same gene and some probes may have no gene annotation (missing values) and therefore can't be tested against a gene set. 

In [15]:
gpl = GEOparse.get_GEO(geo="GPL570", destdir="../data/raw/")
print(gpl.table.shape)
print(gpl.table.columns.tolist())
print('duplicate gene symbols:', gpl.table['Gene Symbol'].duplicated().sum())
print('missing gene symbols:', gpl.table['Gene Symbol'].isna().sum())


21-Aug-2026 09:32:01 DEBUG utils - Directory ../data/raw/ already exists. Skipping.
21-Aug-2026 09:32:01 INFO GEOparse - File already exist: using local version.
21-Aug-2026 09:32:01 INFO GEOparse - Parsing ../data/raw/GPL570.txt: 
21-Aug-2026 09:32:01 DEBUG GEOparse - PLATFORM: GPL570


(54675, 16)
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
duplicate gene symbols: 31154
missing gene symbols: 8893


/Users/joshuasim/Desktop/summer_projects/schizophrenia-pathway-classifier/venv/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Shape is (54675,16) matching typical GPL570 structure. I identified `Gene Symbol` as the column containing gene symbols, and found 31154 duplicate gene symbols and 8893 missing gene symbols. The high duplicate count is expected since multiple probes per gene is common on Affymetrix arrays — not a data quality issue. The missing gene symbols are likely control probes, ESTs, and other unannotated sequences that are included in the array design, and fall within a normal range for GPL570. 

To create a look-up table mapping probe IDs to gene symbols, I will set `gpl_table`'s index to `ID`, then convert the `Gene Symbol` column into a dictionary using `.to_dict()`, giving probe ID to gene symbol pairs. 

In [11]:
gpl_key = gpl.table.set_index('ID')
probe_to_gene = gpl_key['Gene Symbol'].to_dict()

# check mapping with probe ID 224264_x_at (outliers in EDA traced back to this probe)
probe_to_gene['224264_x_at']

'ZAN'

I verify the mapping was correctly done by checking probe `224264_x_at` against what gene symbol it is officially associated to (ZAN, according to biogps) — since `probe_to_gene` returned the correct symbol `ZAN`, it confirms the mapping is accurate. 

## 2.3 Outlier Probe Flag
Since the outlier samples' diagnosis split was ambiguous (5 of the 6 outlier samples were SCZ, not enough to come to a conclusion), I decided to keep probe `224264_x_at`, letting differential expression testing itself determine whether the outliers are statistically significant.

In [16]:
flagged_probes = ['224264_x_at']
# this will be used in DE testing to flag outliers

## 2.4 Differential Expression Testing

I use DE testing to compare the gene activity levels between the SCZ and control samples per probe in order to produce the ranked gene list required for GSEA. I apply Welch's t-test (doesn't require variances to be equal) for determining statistical significance of expression levels, producing a dataframe with test statistics and raw p-value. 

Due to the small sample size and the fragility of single-gene findings in schizophrenia research (found during my literature review) individual DE significance is treated as secondary evidence.

In [17]:
merged_df.head()

,sample_id,duration,diagnosis,name,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,...,AFFX-r2-Bs-dap-M_at,AFFX-r2-Bs-lys-3_at,AFFX-r2-Bs-lys-5_at,AFFX-r2-Bs-lys-M_at,AFFX-r2-Bs-phe-3_at,AFFX-r2-Bs-phe-5_at,AFFX-r2-Bs-phe-M_at,AFFX-r2-Bs-thr-3_s_at,AFFX-r2-Bs-thr-5_s_at,AFFX-r2-Bs-thr-M_s_at
0,GSM528831,short,control,GSM528831,10.0887,7.0144,4.8340,7.9776,5.3040,7.5433,...,9.3438,6.1635,5.6638,6.1582,7.5421,6.5671,6.8474,7.9248,7.7093,7.8684
1,GSM528832,short,control,GSM528832,9.1448,7.0536,5.1611,8.5253,5.8394,7.9697,...,9.2325,5.7333,5.6336,5.6582,7.5044,5.6265,6.6950,7.8944,7.6562,7.8619
2,GSM528833,short,control,GSM528833,9.5847,6.9642,4.9111,8.1324,5.3994,7.8161,...,8.3691,4.8767,4.5944,4.8525,6.5983,5.1199,5.9035,7.1514,6.9449,7.4193
3,GSM528834,short,control,GSM528834,9.2438,7.2024,5.2503,7.6403,5.5726,7.2828,...,8.6710,5.8325,5.5330,5.8352,6.9280,5.9447,6.1117,7.4252,7.3752,7.4043
4,GSM528835,short,control,GSM528835,9.0333,6.6731,4.8153,7.4291,5.3081,7.1056,...,8.9114,5.8716,5.4419,5.2717,7.3429,6.0414,6.6683,8.0693,7.6542,7.7834


In [18]:
merged_df['diagnosis'].unique()

<StringArray>
['control', 'schizophrenia']
Length: 2, dtype: str

In [19]:
# split merged_df into control and scz groups
control_split = merged_df[merged_df['diagnosis'] == 'control']
scz_split = merged_df[merged_df['diagnosis'] == 'schizophrenia']

# DE testing
de = []
for probe in merged_df.drop(columns=['sample_id', 'duration','diagnosis', 'name']).columns:
    
    control_val = control_split[probe].values
    scz_val = scz_split[probe].values
    
    t_stat, p_val = stats.ttest_ind(control_val, scz_val, equal_var=False)
    de.append({'probe': probe, 't_stat': t_stat, 'p_val': p_val}) 

# convert de list to df
DE_df = pd.DataFrame(de)

# flag outliers
DE_df['flagged'] = DE_df['probe'].isin(flagged_probes)

# checks
print(DE_df.shape)
print(DE_df[DE_df['probe'] == '224264_x_at'])
print(DE_df[['t_stat', 'p_val']].isna().sum())


(30061, 4)
             probe    t_stat     p_val  flagged
17601  224264_x_at -1.601338  0.116216     True
t_stat    0
p_val     0
dtype: int64


I check the dataframe was constructed correctly, verifying shape (30061, 4), spot checking `224264_x_at` probe was flagged true, and no missing values for `t_stat`s/`p_val`s. 

## 2.5 FDR — Multiple Testing Correction

Because of the large number of t-tests produced (30061 probes), there is a high potential for false positives (30061 x 5% false positive rate/test = 1500 false positives), so FDR multiple testing correction fixes this by controlling the expected proportion of statistically significant values that are false discoveries. 

I use `fdr_bh` (Benjamini-Hochberg) since it assumes independent or positively correlated tests, and set `alpha=0.05` as the threshold since `0.05` is the standard threshold in genomics DE analysis.

In [21]:
rejected, corrected_p_vals, _, _ = smt.multipletests(DE_df['p_val'], alpha=0.05, method='fdr_bh')

# add corrected p-values to DE_df
DE_df['corrected_p_val'] = corrected_p_vals

# flag significant probes
DE_df['significant'] = rejected

# checks
print(DE_df['significant'].sum())
print(DE_df[DE_df['probe'] == '224264_x_at']['significant'])

0
17601    False
Name: significant, dtype: bool


Looking at the number of significant values in `DE_df` (0), shows that individual genes do not show significant differential expression. This further supports the need for GSEA since the individual-gene approach does not work.

## 2.6 Rank Gene List for GSEA

To rank gene list for GSEA, I apply the `probe_to_gene` lookup dictionary created in 2.2 to the `DE_df`, ranking based on t-statistic since it carries both magnitude and direction. To deal with multiple probes that have the same gene symbol (31154 found in 2.2) I take the probe with the largest absolute value since it is commonly used in GSEA pipelines — prevents the potential of washing out real signal by averaging. To handle the 8893 missing gene symbols found in 2.2 I simply drop those specific rows. 


In [ ]:
# map probe to gene symbol
DE_df['gene symbol'] = DE_df['probe'].map(probe_to_gene) 
print('shape of DE_df after map ', DE_df.shape)

# drop row with missing gene symbols
DE_df = DE_df.dropna(subset=['gene symbol'])
print('shape of DE_df after drop missing', DE_df.shape)

# create column of absolute t-stats, then use it to sort and drop duplicates
DE_df['abs_t_stat'] = DE_df['t_stat'].abs()
DE_df = DE_df.sort_values('abs_t_stat', ascending=False).drop_duplicates(subset='gene symbol', keep='first')

print('shape of DE_df after sort and drop duplicates', DE_df.shape)

# create ranked gene list
ranked_genes = DE_df.sort_values('t_stat', ascending=False)
ranked_genes.shape

shape of DE_df after map  (30061, 7)
shape of DE_df after drop missing (25162, 7)
shape of DE_df after sort and drop duplicates (15706, 8)


(15706, 8)

After mapping `probe_to_gene` to `DE_df`, dropping 4899 unmapped probes, and collapsing 9456 duplicate gene-symbol rows, the final ranked gene list has 15706 unique genes. This will be the input for GSEA to test for pathway-level enrichment. 

## 2.7 GSEA Against Immune/Inflammatory Gene Set

## 2.8 Hypothesis 1 answering

## 2.9 Summary